# 猪特征工程 - 特征提取

标签：无猪 -0，活猪 -1，死猪 -2

原始特征：
- max_channel: 雷达检测到最大通道
- ideal_channel: 理想通道
- br_amp: 呼吸振幅（1 小时内提取的列表）
- br_cv: 呼吸的波动率（1 小时内提取的列表）
- up_down: 是否起卧（1 小时内提取的列表）

In [1]:
import pandas as pd
import numpy as np

# 加载已分割的训练集和测试集
df_train = pd.read_pickle('pig_features_data_train.pkl')
df_test = pd.read_pickle('pig_features_data_test.pkl')
df_test1 = pd.read_pickle('pig_features_data_all_train_houbei.pkl')
print(f"训练集形状：{df_train.shape}")
print(f"测试集形状：{df_test.shape}")
print(f"训练集标签分布：无猪 -0: {(df_train['label']==0).sum()}, 活猪 -1: {(df_train['label']==1).sum()}, 死猪 -2: {(df_train['label']==2).sum()}")
print(f"测试集标签分布：无猪 -0: {(df_test['label']==0).sum()}, 活猪 -1: {(df_test['label']==1).sum()}, 死猪 -2: {(df_test['label']==2).sum()}")

训练集形状：(734659, 8)
测试集形状：(183665, 8)
训练集标签分布：无猪 -0: 6610, 活猪 -1: 727989, 死猪 -2: 60
测试集标签分布：无猪 -0: 1652, 活猪 -1: 181998, 死猪 -2: 15


In [ ]:
# df_test1 = df_test1[df_test1['file_path'].str.contains('houbei', na=False)].copy()

In [2]:
def extract_features_for_segment(br_amp_segment, br_cv_segment, max_ch_segment, ideal_ch_segment, up_down_segment):
    """从单个时间片段中提取特征"""
    features = {}
    
    # ============ 1. 呼吸振幅 (br_amp) 特征 ============
    features['br_amp_mean'] = np.mean(br_amp_segment)  # 平均呼吸振幅
    features['breathing_stop_ratio'] = np.mean(br_amp_segment < 0.2)  # 呼吸停止检测
    features['breathing_high_ratio'] = np.mean(br_amp_segment > 1.0)  # 高呼吸检测
    
    # ============ 2. 呼吸波动率 (br_cv) 特征 ============
    features['br_cv_mean'] = np.mean(br_cv_segment)
    features['br_cv_high_ratio'] = np.mean(br_cv_segment > 1.0)  # 高波动率占比
    features['br_cv_mid_ratio'] = np.mean((0.5 < br_cv_segment) & (br_cv_segment < 1))
    features['br_cv_low_ratio'] = np.mean(br_cv_segment < 0.5)   # 低波动率占比
    
    # ============ 3. 通道相关特征 ============
    channel_match = (max_ch_segment == ideal_ch_segment).astype(float)
    features['channel_match_ratio'] = np.mean(channel_match)  # 通道匹配率
    max_ch_diff = np.diff(max_ch_segment)
    features['max_channel_changes'] = np.sum(max_ch_diff != 0)  # 通道变化次数
    features['max_channel_no'] = np.mean((max_ch_segment >= 21) & (max_ch_segment <= 23)) #无猪通道
    features['max_channel_dead'] = np.mean((max_ch_segment >= 12) &(max_ch_segment <= 14)) #死猪通道
    features['max_channel_13'] = np.mean(max_ch_segment == 13) #无猪/有猪通道
    features['ideal_channel_13'] = np.mean(ideal_ch_segment == 13) 
    
    # ============ 4. 起卧行为特征 ============
    features['up_down_ratio'] = np.mean(up_down_segment)  # 起卧时间占比
    up_down_diff = np.abs(np.diff(up_down_segment))
    features['up_down_transitions'] = np.sum(up_down_diff > 0.5)  # 状态转换次数
    
    # 连续起卧时长
    consecutive_lengths = []
    current_length = 0
    for val in up_down_segment:
        if val > 0.5:
            current_length += 1
        else:
            if current_length > 0:
                consecutive_lengths.append(current_length)
            current_length = 0
    if current_length > 0:
        consecutive_lengths.append(current_length)
    features['up_down_avg_duration'] = np.mean(consecutive_lengths) if consecutive_lengths else 0
    
    # ============ 5. 组合特征 ============
    features['activity_score'] = features['up_down_ratio'] * features['max_channel_changes']
    features['breathing_stability'] = features['br_amp_mean'] * (1 - features['br_cv_high_ratio'])
    
    # ============ 6. 基于规则分类的比例特征 ============
    rule_rates = calc_rule_rates_for_segment(max_ch_segment, br_amp_segment, up_down_segment)
    features['rate_alive'] = rule_rates['rate_alive']
    features['rate_dead'] = rule_rates['rate_dead']
    features['rate_empty'] = rule_rates['rate_empty']
    
    return features

def calc_rule_rates_for_segment(mc_arr, amp_arr, up_arr):
    """基于规则的分类比例计算（针对单个片段）"""
    n = len(mc_arr)
    if n == 0:
        return {'rate_alive': 0, 'rate_dead': 0, 'rate_empty': 0}

    # 条件掩码
    # 活猪：(mc <= 20 and amp >= 0.5) or is_up
    mask_alive = ((mc_arr <= 20) & (amp_arr >= 0.5)) | (up_arr == 1)
    
    # 死猪：13 <= mc <= 15 and amp < 0.5
    mask_dead = (mc_arr >= 13) & (mc_arr <= 15) & (amp_arr < 0.5)
    
    # 无猪：其余情况
    mask_empty = ~(mask_alive | mask_dead)

    return {
        'rate_alive': np.sum(mask_alive) / n,
        'rate_dead': np.sum(mask_dead) / n,
        'rate_empty': np.sum(mask_empty) / n
    }

def extract_features_with_segments(row, segment_size=30):
    """将样本分割成多个片段并提取每个片段的特征"""
    # 获取各个特征数组
    max_ch = np.array(row['max_channel'])
    ideal_ch = np.array(row['ideal_channel'])
    br_amp = np.array(row['br_amp'])
    br_cv = np.array(row['br_cv'])
    up_down = np.array(row['up_down'])
    
    total_points = len(br_amp)
    num_segments = total_points // segment_size  # 分割成4个片段
    
    segments_features = []
    
    for i in range(num_segments):
        start_idx = i * segment_size
        end_idx = (i + 1) * segment_size
        
        # 提取当前片段
        br_amp_seg = br_amp[start_idx:end_idx]
        br_cv_seg = br_cv[start_idx:end_idx]
        max_ch_seg = max_ch[start_idx:end_idx]
        ideal_ch_seg = ideal_ch[start_idx:end_idx]
        up_down_seg = up_down[start_idx:end_idx]
        
        # 提取特征
        features = extract_features_for_segment(
            br_amp_seg, br_cv_seg, max_ch_seg, ideal_ch_seg, up_down_seg
        )
        features['segment_id'] = i  # 片段ID: 0, 1, 2, 3
        features['label'] = row['label']
        features['file_path'] = row['file_path']
        segments_features.append(features)
    
    return segments_features

In [ ]:
# 提取训练集特征（每个样本分割成 4 个 15 分钟片段）
print("正在提取训练集特征（15 分钟分割）...")
train_features = []
for idx, row in df_train.iterrows():
    segment_features = extract_features_with_segments(row, segment_size=30)
    train_features.extend(segment_features)
    if (idx + 1) % 100 == 0:
        print(f"已处理 {idx + 1}/{len(df_train)} 训练样本，当前特征行数：{len(train_features)}")

train_features_df = pd.DataFrame(train_features)

# 提取测试集特征（每个样本分割成 4 个 15 分钟片段）
print("\n正在提取测试集特征（15 分钟分割）...")
test_features = []
for idx, row in df_test.iterrows():
    segment_features = extract_features_with_segments(row, segment_size=30)
    test_features.extend(segment_features)
    if (idx + 1) % 100 == 0:
        print(f"已处理 {idx + 1}/{len(df_test)} 测试样本，当前特征行数：{len(test_features)}")

test_features_df = pd.DataFrame(test_features)

test1_features = []
for idx, row in df_test1.iterrows():
    segment_features = extract_features_with_segments(row, segment_size=30)
    test1_features.extend(segment_features)
    if (idx + 1) % 100 == 0:
        print(f"已处理 {idx + 1}/{len(df_test1)} 测试样本，当前特征行数：{len(test1_features)}")

test1_features_df = pd.DataFrame(test1_features)

print(f"\n训练集特征形状（片段级）：{train_features_df.shape}")
print(f"测试集特征形状（片段级）：{test_features_df.shape}")


print(f"原始样本数：训练集={len(df_train)}, 测试集={len(df_test)}")
print(f"每个样本分割成 4 个片段，总片段数：训练集={len(train_features_df)}, 测试集={len(test_features_df)}")

正在提取训练集特征（15 分钟分割）...
已处理 872600/734659 训练样本，当前特征行数：232
已处理 689000/734659 训练样本，当前特征行数：1344
已处理 665100/734659 训练样本，当前特征行数：1376
已处理 760600/734659 训练样本，当前特征行数：1484
已处理 826900/734659 训练样本，当前特征行数：1944
已处理 146400/734659 训练样本，当前特征行数：2136
已处理 614100/734659 训练样本，当前特征行数：2432
已处理 909300/734659 训练样本，当前特征行数：2560
已处理 837500/734659 训练样本，当前特征行数：2580
已处理 560900/734659 训练样本，当前特征行数：2740
已处理 27200/734659 训练样本，当前特征行数：3204
已处理 264100/734659 训练样本，当前特征行数：3232
已处理 338900/734659 训练样本，当前特征行数：3680
已处理 654400/734659 训练样本，当前特征行数：3700
已处理 439800/734659 训练样本，当前特征行数：4264
已处理 416100/734659 训练样本，当前特征行数：4440
已处理 569300/734659 训练样本，当前特征行数：4892
已处理 5800/734659 训练样本，当前特征行数：4900
已处理 128900/734659 训练样本，当前特征行数：6656
已处理 676700/734659 训练样本，当前特征行数：7552
已处理 696900/734659 训练样本，当前特征行数：7880
已处理 289800/734659 训练样本，当前特征行数：8304
已处理 823200/734659 训练样本，当前特征行数：8532
已处理 666200/734659 训练样本，当前特征行数：9056
已处理 163700/734659 训练样本，当前特征行数：9784
已处理 579000/734659 训练样本，当前特征行数：10968
已处理 152900/734659 训练样本，当前特征行数：12644
已处理 156600/734659 训练样本，当前特征行数：13360

MemoryError: 

In [ ]:
# 特征与标签的相关性分析（基于片段级训练集）
from scipy.stats import pearsonr

print("\n=== 特征与标签的相关性（片段级，Top 20）===")
feature_cols_analysis = [c for c in train_features_df.columns if c not in ['label', 'file_path', 'segment_id']]
correlations = []
for col in feature_cols_analysis:
    corr, _ = pearsonr(train_features_df[col], train_features_df['label'])
    correlations.append((col, corr))

correlations.sort(key=lambda x: abs(x[1]), reverse=True)
print(f"{'特征':<30} {'相关系数':>10}")
print('-' * 45)
for col, corr in correlations[:20]:
    print(f"{col:<30} {corr:>10.4f}")

In [5]:
# ==================== XGBoost 模型训练 ====================
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# 准备数据
feature_cols = [c for c in train_features_df.columns if c not in ['label', 'file_path', 'segment_id']]
X_train = train_features_df[feature_cols].values
y_train = train_features_df['label'].values

X_test = test_features_df[feature_cols].values
y_test = test_features_df['label'].values

X_test1 = test1_features_df[feature_cols].values
y_test1 = test1_features_df['label'].values

file_paths_test = test_features_df['file_path'].values
segment_ids_test = test_features_df['segment_id'].values

print(f"训练集片段数：{len(X_train)}, 测试集片段数：{len(X_test)}")
print(f"训练集标签分布：0={sum(y_train==0)}, 1={sum(y_train==1)}, 2={sum(y_train==2)}")
print(f"测试集标签分布：0={sum(y_test==0)}, 1={sum(y_test==1)}, 2={sum(y_test==2)}")
print(f"测试集1标签分布：0={sum(y_test1==0)}, 1={sum(y_test1==1)}, 2={sum(y_test1==2)}")

训练集片段数：278524, 测试集片段数：69632
训练集标签分布：0=26440, 1=251844, 2=240
测试集标签分布：0=6608, 1=62964, 2=60
测试集1标签分布：0=0, 1=871204, 2=0


In [6]:
# 训练 XGBoost 模型
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss',
    subsample = 0.6,
    colsample_bytree = 0.9
)

model.fit(X_train, y_train)

# 预测 - 获取每个片段的预测结果和预测概率
y_pred_segments = model.predict(X_test)
y_proba_segments = model.predict_proba(X_test)  # 获取预测概率

# 获取唯一文件路径
unique_files = np.unique(file_paths_test)
print(f"\n测试集唯一样本数：{len(unique_files)}")

d:\miniforge\envs\pig\lib\site-packages\xgboost\core.py:158: UserWarning: [15:14:44] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



测试集唯一样本数：17408


In [7]:
# ==================== 融合算法 ====================
# 对每个样本的 4 个片段预测结果进行融合
# 融合策略：投票法 + 概率加权

def fuse_predictions(file_paths, segment_preds, segment_probas, true_labels):
    """
    融合每个样本的 4 个片段预测结果
    
    融合算法：
    1. 投票法：4 个片段投票，多数票决定最终结果
    2. 概率平均：计算每个类别的平均概率，选择概率最高的类别
    3. 加权投票：根据片段 ID 赋予不同权重（越靠近后面的片段权重越高）
    """
    unique_files = np.unique(file_paths)
    fused_preds = []
    fused_proba_max = []
    true_labels_fused = []
    
    for file_path in unique_files:
        # 获取该文件的所有片段
        mask = file_paths == file_path
        file_preds = segment_preds[mask]
        file_probas = segment_probas[mask]
        file_true = true_labels[mask][0]  # 同一样本的标签相同
        
        # 方法 1: 投票法（修改版）- 要求大于总数的0.4且最大
        votes = np.bincount(file_preds, minlength=3)
        total_votes = len(file_preds)
        threshold = total_votes * 0.4

        # 找到最大票数和对应的类别
        max_votes = np.max(votes)
        max_vote_class = np.argmax(votes)

        # 检查是否满足条件：大于阈值且最大
        if max_votes > threshold:
            vote_pred = max_vote_class
        else:
            vote_pred = None  # 第一种方法失效

        # 方法 2: 概率平均法（保持不变）
        avg_proba = np.mean(file_probas, axis=0)
        proba_pred = np.argmax(avg_proba)

        # 方法 3: 融合投票法和概率平均法（第一种失效用第二种）
        if vote_pred is not None:
            # 第一种方法有效时使用投票法结果
            weighted_pred = vote_pred
        else:
            # 第一种方法失效时使用概率平均法结果
            weighted_pred = proba_pred

        
        # 最终融合策略：
        final_pred = weighted_pred
        
        fused_preds.append(final_pred)
        fused_proba_max.append(np.max(avg_proba))  # 最大概率值（置信度）
        true_labels_fused.append(file_true)
    
    return np.array(fused_preds), np.array(true_labels_fused), np.array(fused_proba_max)

# 应用融合算法
y_pred_fused, y_test_fused, confidence_scores = fuse_predictions(
    file_paths_test, y_pred_segments, y_proba_segments, y_test
)


In [8]:

# ==================== 直接结果混淆矩阵（不融合，每个分片就是一个样本）==================
print("=" * 60)
print("=== 直接结果混淆矩阵（不融合，每个分片就是一个样本）===")
print("=" * 60)

# 计算直接预测的准确率（每个片段独立）
accuracy_direct = accuracy_score(y_test, y_pred_segments)
print(f"\n直接预测测试集准确率：{accuracy_direct:.4f}")

# 混淆矩阵（直接结果，不融合）
cm_direct = confusion_matrix(y_test, y_pred_segments)
print(f"\n=== 混淆矩阵 (直接结果，不融合) ===")
print("预测 →")
print("真实 ↓    无猪 (0)  活猪 (1)  死猪 (2)")
print(f"无猪 (0)    {cm_direct[0][0]:^5}   {cm_direct[0][1]:^5}   {cm_direct[0][2]:^5}")
print(f"活猪 (1)    {cm_direct[1][0]:^5}   {cm_direct[1][1]:^5}   {cm_direct[1][2]:^5}")
print(f"死猪 (2)    {cm_direct[2][0]:^5}   {cm_direct[2][1]:^5}   {cm_direct[2][2]:^5}")

# 分类报告（直接结果）
print("\n=== 分类报告 (直接结果，不融合) ===")
print(classification_report(y_test, y_pred_segments, target_names=['无猪 (0)', '活猪 (1)', '死猪 (2)']))



=== 直接结果混淆矩阵（不融合，每个分片就是一个样本）===

直接预测测试集准确率：0.9986

=== 混淆矩阵 (直接结果，不融合) ===
预测 →
真实 ↓    无猪 (0)  活猪 (1)  死猪 (2)
无猪 (0)    6563     45       0  
活猪 (1)     46     62917     1  
死猪 (2)      0       6      54  

=== 分类报告 (直接结果，不融合) ===
              precision    recall  f1-score   support

      无猪 (0)       0.99      0.99      0.99      6608
      活猪 (1)       1.00      1.00      1.00     62964
      死猪 (2)       0.98      0.90      0.94        60

    accuracy                           1.00     69632
   macro avg       0.99      0.96      0.98     69632
weighted avg       1.00      1.00      1.00     69632



In [9]:
y_pred_segments1 = model.predict(X_test1)
accuracy_direct = accuracy_score(y_test1, y_pred_segments1)
print(f"\n直接预测测试集准确率：{accuracy_direct:.4f}")

# 混淆矩阵（直接结果，不融合）
cm_direct = confusion_matrix(y_test1, y_pred_segments1)
print(f"\n=== 混淆矩阵 (直接结果，不融合) ===")
print("预测 →")
print("真实 ↓    无猪 (0)  活猪 (1)  死猪 (2)")
print(f"无猪 (0)    {cm_direct[0][0]:^5}   {cm_direct[0][1]:^5}   {cm_direct[0][2]:^5}")
print(f"活猪 (1)    {cm_direct[1][0]:^5}   {cm_direct[1][1]:^5}   {cm_direct[1][2]:^5}")
print(f"死猪 (2)    {cm_direct[2][0]:^5}   {cm_direct[2][1]:^5}   {cm_direct[2][2]:^5}")


直接预测测试集准确率：0.9584

=== 混淆矩阵 (直接结果，不融合) ===
预测 →
真实 ↓    无猪 (0)  活猪 (1)  死猪 (2)
无猪 (0)      0       0       0  
活猪 (1)    36187   834963    54  
死猪 (2)      0       0       0  


In [ ]:
def analyze_prediction_errors(y_pred, features_df, top_n=20):
    """
    分析预测错误的样本，提取雷达号和日期，判断错误是否集中在特定雷达或特定日期。

    Parameters:
    -----------
    y_pred : array-like
        模型的预测结果数组。
    features_df : pd.DataFrame
        包含特征和真实标签的数据框，必须包含 'file_path' 和 'label' 列。
    top_n : int, optional (default=20)
        显示错误最多的雷达号数量。

    Returns:
    --------
    dict
        包含以下键的字典：
        - 'error_df': 预测错误的样本详细数据（含 radar_id, date）。
        - 'error_stats_radar': 按雷达号统计的错误次数。
        - 'error_stats_date': 按日期统计的错误次数。
        - 'error_stats_radar_date': 雷达 x 日期的交叉统计表。
        - 'summary': 包含关键统计指标的字典。
    """
    
    # 1. 数据校验
    if len(y_pred) != len(features_df):
        raise ValueError(f"预测结果长度 ({len(y_pred)}) 与数据长度 ({len(features_df)}) 不一致！")
    
    if 'file_path' not in features_df.columns or 'label' not in features_df.columns:
        raise ValueError("features_df 中必须包含 'file_path' 和 'label' 列")

    df = features_df.copy()
    
    # 2. 提取辅助信息 (雷达号 & 日期)
    def extract_info(path):
        if not isinstance(path, str):
            return None, None
        
        # --- 提取雷达号 ---
        filename = path.split('/')[-1] # 例如：30029-2025050708.hdf5
        parts = filename.split('-')
        radar_id = parts[0] if len(parts) > 0 else None
        
        # --- 提取日期 ---
        date_str = None
        # 策略 A: 尝试从文件名中提取 (例如 30029-2025050708.hdf5 -> 20250507)
        # 假设文件名格式为：{radar}-{YYYYMMDD}{HH}.hdf5
        if len(parts) > 1:
            time_part = parts[1] # 例如 2025050708.hdf5
            # 去除后缀
            time_part_clean = time_part.split('.')[0] 
            # 取前8位作为日期
            if len(time_part_clean) >= 8 and time_part_clean[:8].isdigit():
                date_str = time_part_clean[:8]
        
        # 策略 B: 如果文件名没解析出来，尝试从路径文件夹中找 (例如 .../20250507/...)
        if not date_str:
            # 使用正则查找路径中连续的8位数字
            match = re.search(r'(\d{8})', path)
            if match:
                date_str = match.group(1)
                
        return radar_id, date_str

    # 应用提取
    extracted = df['file_path'].apply(extract_info)
    df['radar_id'] = [x[0] for x in extracted]
    df['date'] = [x[1] for x in extracted]
    df['pred_label'] = y_pred

    # 3. 筛选错误样本
    error_df = df[df['pred_label'] != df['label']].copy()
    
    if len(error_df) == 0:
        print("恭喜！没有预测错误的样本。")
        return {
            'error_df': pd.DataFrame(),
            'error_stats_radar': pd.DataFrame(),
            'error_stats_date': pd.DataFrame(),
            'error_stats_radar_date': pd.DataFrame(),
            'summary': {'total_errors': 0, 'is_concentrated': False}
        }

    # 4. 统计分析
    
    # 4.1 按雷达统计
    error_stats_radar = error_df['radar_id'].value_counts().reset_index()
    error_stats_radar.columns = ['radar_id', 'error_count']
    
    # 4.2 按日期统计 (新增)
    error_stats_date = error_df['date'].value_counts().reset_index()
    error_stats_date.columns = ['date', 'error_count']
    # # 按日期排序
    # error_stats_date = error_stats_date.sort_values('error_count').reset_index(drop=True)

    # 4.3 雷达 x 日期 交叉统计 (新增)
    # 行：雷达，列：日期，值：错误次数
    error_stats_radar_date = pd.crosstab(error_df['radar_id'], error_df['date'])
    
    # 5. 计算集中度指标 (仍基于雷达)
    total_errors = len(error_df)
    top_radar_row = error_stats_radar.iloc[0]
    top_radar_id = top_radar_row['radar_id']
    top_radar_count = top_radar_row['error_count']
    concentration_ratio = top_radar_count / total_errors

    
    # 6. 打印分析报告
    print(f"总样本数：{len(df)}")
    print(f"预测错误样本数：{total_errors}")
    print(f"错误率：{total_errors / len(df):.2%}")
    
    print(f"\n--- [雷达维度] 错误分布 (Top {top_n}) ---")
    print(error_stats_radar.head(top_n).to_string(index=False))
    
    print(f"\n--- [日期维度] 错误分布 (Top {top_n}) ---")
    print(error_stats_date.head(top_n).to_string(index=False))
    
    print(f"\n--- [交叉分析] 错误最多的雷达/日期组合 (Top 10) ---")
    # 将交叉表堆叠，找出最大值组合
    stack_df = error_stats_radar_date.stack().reset_index(name='count')
    stack_df = stack_df.sort_values('count', ascending=False)
    print(stack_df.head(10).to_string(index=False))

    print(f"\n--- 集中度分析 ---")
    print(f"错误最多的雷达：{top_radar_id} (错误次数：{top_radar_count})")
    print(f"该雷达占总错误数的比例：{concentration_ratio:.2%}")
    
    # 检查是否有特定日期错误激增
    if not error_stats_date.empty:
        max_date_row = error_stats_date.loc[error_stats_date['error_count'].idxmax()]
        print(f"错误最多的日期：{max_date_row['date']} (错误次数：{max_date_row['error_count']})")
        

    # 7. 返回结果
    summary = {
        'total_errors': int(total_errors),
        'top_radar_id': str(top_radar_id),
        'top_radar_count': int(top_radar_count),
        'top_date': str(error_stats_date.iloc[-1]['date']) if not error_stats_date.empty else None, # 假设按日期排序后最后一个可能是最新，或者直接取max
        'concentration_ratio': float(concentration_ratio),
    }

    return {
        'error_df': error_df,
        'error_stats_radar': error_stats_radar,
        'error_stats_date': error_stats_date,
        'error_stats_radar_date': error_stats_radar_date,
        'summary': summary
    }

In [16]:
# 1. 调用分析函数
results = analyze_prediction_errors(y_pred_segments1, test1_features_df)

总样本数：871204
预测错误样本数：36241
错误率：4.16%

--- [雷达维度] 错误分布 (Top 20) ---
radar_id  error_count
   27221          899
   26442          879
   13656          696
   26935          653
   13655          645
   26183          633
   26488          424
   25394          393
   13652          381
   26708          368
   13634          363
   13670          356
   25396          336
   13685          322
   25652          310
   13122          301
   26456          300
   13138          289
   12886          286
   14701          284

--- [日期维度] 错误分布 (Top 20) ---
    date  error_count
20251029         3840
20251028         2138
20260104         1580
20260105         1468
20250922         1233
20250510         1127
20260103         1019
20251231          958
20251218          953
20251214          776
20251213          754
20251217          741
20260101          736
20260102          704
20251215          676
20251216          662
20250605          613
20250509          601
20250604          538
20

In [ ]:
results['error_stats_radar_date'].to_csv('error_stats_radar_date_matrix.csv', encoding='utf-8-sig')

In [ ]:
df_output = pd.DataFrame({'full_file_path': results['error_df']['file_path']})
output_filename = 'error_paths.csv'
df_output.to_csv(output_filename, index=False, encoding='utf-8-sig')

In [ ]:

# 计算融合后的准确率
accuracy_fused = accuracy_score(y_test_fused, y_pred_fused)
print(f"\n=== 融合后结果 ===")
print(f"融合后测试集准确率：{accuracy_fused:.4f}")

# 混淆矩阵（融合后）
cm_fused = confusion_matrix(y_test_fused, y_pred_fused)
print(f"\n=== 混淆矩阵 (融合后) ===")
print("预测 →")
print("真实 ↓    无猪 (0)  活猪 (1)  死猪 (2)")
print(f"无猪 (0)    {cm_fused[0][0]:^5}   {cm_fused[0][1]:^5}   {cm_fused[0][2]:^5}")
print(f"活猪 (1)    {cm_fused[1][0]:^5}   {cm_fused[1][1]:^5}   {cm_fused[1][2]:^5}")
print(f"死猪 (2)    {cm_fused[2][0]:^5}   {cm_fused[2][1]:^5}   {cm_fused[2][2]:^5}")

# 分类报告
print("\n=== 分类报告 (融合后) ===")
print(classification_report(y_test_fused, y_pred_fused, target_names=['无猪 (0)', '活猪 (1)', '死猪 (2)']))

In [ ]:
# ==================== 混淆矩阵可视化 ====================

# 绘制融合后的混淆矩阵
# plt.figure(figsize=(10, 8))

label_names = ['无猪 (0)', '活猪 (1)', '死猪 (2)']

# # 计算百分比
# cm_percent = cm_fused.astype('float') / cm_fused.sum(axis=1)[:, np.newaxis] * 100

# # 绘制热力图
# sns.heatmap(cm_fused, annot=True, fmt='d', cmap='Blues', 
#             xticklabels=label_names, yticklabels=label_names,
#             cbar=True, square=True, linewidths=1, linecolor='gray')

# plt.title('Confusion Matrix (Fused Predictions - 15min Segments)', fontsize=14)
# plt.ylabel('True Label', fontsize=12)
# plt.xlabel('Predicted Label', fontsize=12)

# # 在每个格子中添加百分比
# for i in range(len(cm_fused)):
#     for j in range(len(cm_fused[0])):
#         plt.text(j + 0.5, i + 0.5, f'{cm_percent[i, j]:.1f}%', 
#                 ha='center', va='center', color='gray', fontsize=10)

# plt.tight_layout()
# plt.savefig('confusion_matrix_fused.png', dpi=150)
# plt.show()

# # ==================== 各片段混淆矩阵对比 ====================
# fig, axes = plt.subplots(2, 2, figsize=(16, 12))
# axes = axes.flatten()

# for seg_id in range(4):
#     mask = segment_ids_test == seg_id
#     cm_seg = confusion_matrix(y_test[mask], y_pred_segments[mask])
    
#     sns.heatmap(cm_seg, annot=True, fmt='d', cmap='Greens', 
#                 xticklabels=label_names, yticklabels=label_names,
#                 ax=axes[seg_id], cbar=True, square=True, linewidths=1, linecolor='gray')
    
#     axes[seg_id].set_title(f'Segment {seg_id + 1} (0-15 min)' if seg_id == 0 else 
#                           f'Segment {seg_id + 1} (15-30 min)' if seg_id == 1 else
#                           f'Segment {seg_id + 1} (30-45 min)' if seg_id == 2 else
#                           f'Segment {seg_id + 1} (45-60 min)', fontsize=12)
#     axes[seg_id].set_ylabel('True Label', fontsize=10)
#     axes[seg_id].set_xlabel('Predicted Label', fontsize=10)

# plt.suptitle('Confusion Matrices for Each 15-min Segment', fontsize=14, y=1.02)
# plt.tight_layout()
# plt.savefig('confusion_matrix_segments.png', dpi=150)
# plt.show()

# ==================== 获取混淆矩阵每个格子对应的样本 ====================
samples = defaultdict(lambda: defaultdict(list))

for i, (true_label, pred_label, file_path) in enumerate(zip(y_test_fused, y_pred_fused, unique_files)):
    samples[true_label][pred_label].append(file_path)

# 打印每个格子对应的样本数量
print("=== 混淆矩阵样本列表 (融合后) ===\n")
for true_label in sorted(samples.keys()):
    for pred_label in sorted(samples[true_label].keys()):
        file_list = samples[true_label][pred_label]
        print(f"真实:{label_names[true_label]} → 预测:{label_names[pred_label]}")
        print(f"  数量：{len(file_list)}")
        if len(file_list) <= 10:
            print(f"  文件路径：{file_list}\n")
        else:
            print(f"  文件路径 (前 10 个): {file_list[:10]}...\n")

In [ ]:
unique_files = np.unique(file_paths_test)
results_dict = {}

# 用于统计混淆矩阵的计数器
# 结构：confusion_matrix[真实标签][预测标签] = 数量
confusion_matrix_stats = defaultdict(lambda: defaultdict(int))

# 标签映射说明 (用于打印输出)
label_map = {0: "无猪", 1: "有猪", 2: "死猪"}

for file_path in unique_files:
    mask = file_paths_test == file_path
    file_segment_preds = y_pred_segments[mask].tolist()
    file_segment_probas = y_proba_segments[mask].tolist()
    file_true_label = int(y_test[mask][0])
    file_fused_pred = int(y_pred_fused[unique_files.tolist().index(file_path)])

    segment_labels = [int(pred) for pred in file_segment_preds]

    # --- 新增逻辑：判断混淆矩阵类型 ---
    if file_true_label == file_fused_pred:
        confusion_type = "正确分类 (True Positive for this class)"
        description = f"{label_map[file_true_label]} -> {label_map[file_fused_pred]}"
    else:
        confusion_type = "错误分类 (Misclassification)"
        description = f"{label_map[file_true_label]} 被误判为 {label_map[file_fused_pred]}"
    
    # 更新统计数据
    confusion_matrix_stats[file_true_label][file_fused_pred] += 1

    results_dict[file_path] = {
        'true_label': file_true_label,
        'fused_label': file_fused_pred,
        'segment_labels': segment_labels,
        'confusion_type': confusion_type,      # 新增：分类类型
        'description': description             # 新增：人类可读的描述
    }

In [ ]:
results_dict

In [ ]:
import pickle
with open('plot_results.pkl', 'wb') as f:
    pickle.dump(results_dict, f)

In [ ]:
# 保存训练集和测试集特征（片段级）
# train_features_df.to_pickle('engineered_features_segments_train.pkl')
# test_features_df.to_pickle('engineered_features_segments_test.pkl')
# print("\n特征已保存：")
# print(f"  - engineered_features_segments_train.pkl: {train_features_df.shape}")
# print(f"  - engineered_features_segments_test.pkl: {test_features_df.shape}")

# 保存融合后的结果
import pickle
fusion_results = {
    'y_pred_fused': y_pred_fused,
    'y_test_fused': y_test_fused,
    'confidence_scores': confidence_scores,
    'unique_files': unique_files,
    'cm_fused': cm_fused,
    'accuracy_fused': accuracy_fused
}
# with open('fusion_results.pkl', 'wb') as f:
#     pickle.dump(fusion_results, f)
# print(f"  - fusion_results.pkl: 融合后的预测结果")

In [ ]:
# ==================== 绘制死猪样本的一小时趋势图 ====================
# 混淆矩阵各位置对应的文件：
# samples[2][0]: 死猪预测成无猪 (TP+FP组合)
# samples[2][1]: 死猪预测成活猪
# samples[2][2]: 死猪预测成死猪（正确预测）
plt.rcParams['font.sans-serif'] = ['SimHei']  
plt.rcParams['axes.unicode_minus'] = False 
def plot_dead_pig_trends(df_test, samples, feature_name, save_path=None):
    """
    绘制死猪样本的一小时趋势图
    
    参数:
        df_test: 测试集DataFrame
        samples: 混淆矩阵各位置对应的文件字典
        feature_name: 特征名称 ('br_amp', 'br_cv', 'max_ch', 'ideal_ch', 'up_down')
        save_path: 保存路径（可选）
    """
    plt.figure(figsize=(14, 5))
    
    # 映射特征名称到df中的列名
    feature_map = {
        'br_amp': 'br_amp',
        'br_cv': 'br_cv',
        'max_ch': 'max_channel',
        'ideal_ch': 'ideal_channel',
        'up_down': 'up_down'
    }
    col_name = feature_map.get(feature_name, feature_name)
    
    # 三种预测情况
    cases = [
        (0, '死猪预测成无猪', 'red'),
        (1, '死猪预测成活猪', 'orange'),
        (2, '死猪预测成死猪', 'green')
    ]
    
    for pred_label, title, color in cases:
        file_list = samples[2][pred_label]
        
        if len(file_list) == 0:
            print(f"警告: {title} 没有样本")
            continue
            
        # 获取所有样本的数据
        all_data = []
        for file_path in file_list:
            row = df_test[df_test['file_path'] == file_path]
            if len(row) > 0:
                data = row.iloc[0][col_name]
                if isinstance(data, list) or isinstance(data, np.ndarray):
                    all_data.append(np.array(data))
        
        if len(all_data) == 0:
            continue
            
        # 计算平均值和标准差
        # 找到最短的长度
        min_len = min(len(d) for d in all_data)
        all_data_trimmed = [d[:min_len] for d in all_data]
        
        data_array = np.array(all_data_trimmed)
        mean_data = np.mean(data_array, axis=0)
        std_data = np.std(data_array, axis=0)
        
        # 生成时间轴（1小时，假设1分钟一个点，共60个点）
        time_axis = np.arange(len(mean_data))
        
        # 绘制平均曲线
        plt.plot(time_axis, mean_data, label=f'{title} (n={len(all_data)})', 
                 color=color, linewidth=2)
        # 添加置信区间（±1标准差）
        plt.fill_between(time_axis, mean_data - std_data, mean_data + std_data, 
                        color=color, alpha=0.2)
    
    plt.xlabel('时间', fontsize=12)
    plt.ylabel(feature_name, fontsize=12)
    plt.title(f'死猪样本的 {feature_name} 一小时趋势图', fontsize=14)
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"图片已保存至: {save_path}")
    
    plt.show()

# 绘制5张图
print("=" * 60)
print("绘制死猪样本的一小时趋势图")
print("=" * 60)

# 1. br_amp 趋势图
plot_dead_pig_trends(df_test, samples, 'br_amp', 'dead_pig_br_amp_trend.png')

In [ ]:
# 2. br_cv 趋势图
plot_dead_pig_trends(df_test, samples, 'br_cv', 'dead_pig_br_cv_trend.png')

# 3. max_ch 趋势图
plot_dead_pig_trends(df_test, samples, 'max_ch', 'dead_pig_max_ch_trend.png')

# 4. ideal_ch 趋势图
plot_dead_pig_trends(df_test, samples, 'ideal_ch', 'dead_pig_ideal_ch_trend.png')

# 5. up_down 趋势图
plot_dead_pig_trends(df_test, samples, 'up_down', 'dead_pig_up_down_trend.png')

print("\n所有趋势图绘制完成！")

In [ ]:
f_l = ['/data1/chenming40/alive_detect/dead_pig/19762-2025071422.hdf5', '/data1/chenming40/alive_detect/dead_pig/29497-2025090618.hdf5', '/data1/chenming40/alive_detect/dead_pig/29530-2025072120.hdf5']
df_test[df_test['file_path'] == f_l[2]]['up_down'].tolist()